# 🥉 Bronze Layer - Data Ingestion & Data Profiling

## 📌 Objetivo

A camada **Bronze** representa o primeiro estágio do pipeline de dados, sendo responsável pela ingestão e armazenamento dos dados exatamente como foram recebidos da origem, preservando sua integridade e garantindo rastreabilidade para as próximas etapas do ETL.

Nesta fase, nenhuma regra de negócio é aplicada. O foco é armazenar os dados brutos (RAW) e realizar uma primeira análise da qualidade das informações.

---

## 🛠 Tecnologias Utilizadas

- Databricks
- Apache Spark (PySpark)
- SQL
- Delta Lake

---

## 📂 Fontes de Dados

Foram utilizados dois conjuntos de dados:

### Products

Informações sobre produtos comercializados na Amazon.

Principais atributos:

- ASIN
- Título
- Categoria
- Preço
- Avaliação
- Número de Reviews
- Produto Best Seller
- Quantidade comprada no último mês

### Categories

Tabela responsável pelo relacionamento entre o identificador da categoria e seu respectivo nome.

---

## 🎯 Objetivos desta Etapa

- Importar os arquivos para o Databricks
- Criar as tabelas da camada Bronze
- Armazenar os dados em formato Delta
- Validar a estrutura das tabelas
- Analisar a qualidade dos dados
- Identificar valores nulos
- Identificar registros duplicados
- Gerar estatísticas descritivas
- Garantir que os dados estejam prontos para a camada Silver

---

## 📐 Arquitetura

```text
Arquivos de origem
        │
        ▼
Bronze (RAW)
        │
        ▼
Data Profiling
        │
        ▼
Silver
```

---

## 📊 Data Profiling

Durante esta etapa foram realizadas as seguintes validações:

✔ Estrutura das tabelas

✔ Tipos de dados

✔ Contagem de registros

✔ Verificação de valores nulos

✔ Verificação de duplicidades

✔ Estatísticas das colunas numéricas

✔ Validação da integridade entre produtos e categorias

---

## 📦 Tabelas Geradas

Schema:

```text
bronze
```

Tabelas:

```text
bronze.products

bronze.categories
```

---

## ✅ Resultado Esperado

Ao final desta etapa, os dados encontram-se armazenados na camada Bronze, preservando sua estrutura original e prontos para serem transformados na camada Silver, onde serão aplicadas regras de negócio, enriquecimento dos dados e criação de novas métricas analíticas.

In [ ]:
%sql
SELECT COUNT(*) FROM amazon_products;

COUNT(*)
1426337


In [ ]:
%sql
DESCRIBE TABLE amazon_products;

col_name,data_type,comment
asin,string,None
title,string,None
imgUrl,string,None
productURL,string,None
stars,double,None
reviews,bigint,None
price,double,None
listPrice,double,None
category_id,bigint,None
isBestSeller,boolean,None


In [ ]:
from pyspark.sql.functions import col, sum

amazon_products = spark.table("amazon_products")

display(
    amazon_products.select([
        sum(col(c).isNull().cast("int")).alias(c)
        for c in amazon_products.columns
    ])
)

asin,title,imgUrl,productURL,stars,reviews,price,listPrice,category_id,isBestSeller,boughtInLastMonth
0,0,0,0,0,0,0,0,0,0,0


In [ ]:
%sql
SELECT
asin,
COUNT(*)
FROM amazon_products
GROUP BY asin
HAVING COUNT(*) > 1;

asin,COUNT(*)


In [ ]:
from pyspark.sql.functions import count
duplicados = (
    amazon_products
    .groupBy("asin")
    .agg(count("*").alias("qtd"))
    .filter("qtd > 1")
)

display(duplicados)

asin,qtd


In [ ]:
from pyspark.sql.functions import col

display(
    amazon_products.select([
        (col(c) == "").cast("int").alias(c)
        for c in ["title", "imgUrl", "productURL"]
    ])
)

title,imgUrl,productURL
0,0,0
0,0,0
0,0,0
0,0,0
0,0,0
0,0,0
0,0,0
0,0,0
0,0,0
0,0,0


In [ ]:
amazon_products.printSchema()
amazon_categories.printSchema()

root
 |-- asin: string (nullable = true)
 |-- title: string (nullable = true)
 |-- imgUrl: string (nullable = true)
 |-- productURL: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- reviews: long (nullable = true)
 |-- price: double (nullable = true)
 |-- listPrice: double (nullable = true)
 |-- category_id: long (nullable = true)
 |-- isBestSeller: boolean (nullable = true)
 |-- boughtInLastMonth: long (nullable = true)

root
 |-- id: long (nullable = true)
 |-- category_name: string (nullable = true)



In [ ]:
display(
    amazon_products.describe()
)

summary,asin,title,imgUrl,productURL,stars,reviews,price,listPrice,category_id,boughtInLastMonth
count,1426337,1426337,1426337,1426337,1426337,1426337,1426337,1426337,1426337,1426337
mean,1.7831715957285302E9,3.8461538461538465E230,None,None,3.9995118264469034,180.7508197571822,43.375403680996826,12.449159714719398,123.74088101199085,141.98229450683814
stddev,1.8435526727706594E9,NaN,None,None,1.3442923176553623,1761.4529590011518,130.28929583115277,46.111984184646275,73.11273487056731,836.2719652279304
min,0007268149,2pcs LED Light Bar Decal Sticker For controller PS4,No Image Found,https://www.amazon.com/dp/0007268149,0.0,0,0.0,0.0,1,0
max,BT00CTP93I,"🛦 United States Air Force SR-71A Blackbird 8"" Die Cast Metal Pull Back Action Model Jet Toy",https://m.media-amazon.com/images/I/C1Wys+eYwlS._AC_UY218_.jpg,https://www.amazon.com/dp/BT00CTP93I,5.0,346563,19731.81,999.99,270,100000


In [ ]:
amazon_categories = spark.table("amazon_categories")

display(amazon_categories)

id,category_name
1,Beading & Jewelry Making
2,Fabric Decorating
3,Knitting & Crochet Supplies
4,Printmaking Supplies
5,Scrapbooking & Stamping Supplies
6,Sewing Products
7,Craft & Hobby Fabric
8,Needlework Supplies
9,"Arts, Crafts & Sewing Storage"
10,"Painting, Drawing & Art Supplies"


In [ ]:
from pyspark.sql.functions import col

categorias_orfas = (
    amazon_products.alias("p")
    .join(
        amazon_categories.alias("c"),
        col("p.category_id") == col("c.id"),
        "left"
    )
    .filter(col("c.id").isNull())
)

display(categorias_orfas)

asin,title,imgUrl,productURL,stars,reviews,price,listPrice,category_id,isBestSeller,boughtInLastMonth,id,category_name


In [ ]:
display(amazon_products.limit(20))
display(amazon_categories.limit(20))

asin,title,imgUrl,productURL,stars,reviews,price,listPrice,category_id,isBestSeller,boughtInLastMonth
B0BYJPZBV6,"Christian Jesus Cross Ponytail Hat, Embroidered Adjustable Wash Cotton Denim Dad Baseball Cap",https://m.media-amazon.com/images/I/61NNO0EpEsL._AC_UL320_.jpg,https://www.amazon.com/dp/B0BYJPZBV6,4.9,0,18.59,0.0,120,False,0
B00OM8UGZ0,Flower Leaf Faux Pearl Rhinestone Crystal Bridal Tiara Crown T881,https://m.media-amazon.com/images/I/61jakGwNK1L._AC_UL320_.jpg,https://www.amazon.com/dp/B00OM8UGZ0,4.9,0,15.99,0.0,120,False,0
B08VJ5DBGM,Headbands For Women Girls Nurses Doctors With Buttons Elastic Yoga Hair Bands Accessories Non Slip 4 Pcs,https://m.media-amazon.com/images/I/71NFvL87LtL._AC_UL320_.jpg,https://www.amazon.com/dp/B08VJ5DBGM,4.2,0,11.99,0.0,120,False,50
B07R3S1XQK,Women Soft Rayon Snood Hat Hair Net Crocheted Hair Net Cap Mix Colors,https://m.media-amazon.com/images/I/5199hh-YDSL._AC_UL320_.jpg,https://www.amazon.com/dp/B07R3S1XQK,4.1,0,7.99,0.0,120,False,50
B0BKXSTHJB,Reversible Soft Cashmere Feel Scarf Large Shawl Wrap Double-side Art Print Warm Pashmina Klimt Van Gogh Gifts,https://m.media-amazon.com/images/I/815HHUjVi2L._AC_UL320_.jpg,https://www.amazon.com/dp/B0BKXSTHJB,4.7,0,20.99,22.99,120,False,0
B09VNW868C,Wolf/Fox/Cat Ear Cosplay Headwear Girl Accessories Handmade Hairband Furry Ear Faux Fur Headband,https://m.media-amazon.com/images/I/61jaA7qz-KL._AC_UL320_.jpg,https://www.amazon.com/dp/B09VNW868C,4.5,0,12.99,0.0,120,False,0
B00KRG426U,1003 Large Fit Over Safety Glasses Fitover Prescription Eyewear Men Women Wrap Around Tinted Anti Fog Scratch Glare,https://m.media-amazon.com/images/I/61Wk0TFe2SL._AC_UL320_.jpg,https://www.amazon.com/dp/B00KRG426U,4.5,0,8.08,9.0,120,False,50
B07WTJL8GP,Pokémon Characters Sunglasses Costume or Mask with UV400 Protection - One Size Fits Most,https://m.media-amazon.com/images/I/61GB0e4x99L._AC_UL320_.jpg,https://www.amazon.com/dp/B07WTJL8GP,4.6,0,7.99,0.0,120,False,0
B0BZP349L6,Women Wind Coat Waist Belt Replacement Self Tie Dress Pajamas Trench Coat Waistband Belt,https://m.media-amazon.com/images/I/51SGVDjbDlL._AC_UL320_.jpg,https://www.amazon.com/dp/B0BZP349L6,5.0,0,14.99,0.0,120,False,0
B0CB68L179,Sweet 16 Tiara Sweet 16 Sash and Tiara Set Birthday Tiara Birthday Crowns for Women Sweet 16 Crown Sweet Sixteen Birthda,https://m.media-amazon.com/images/I/81y1ryw9k1L._AC_UL320_.jpg,https://www.amazon.com/dp/B0CB68L179,0.0,0,11.99,0.0,120,False,0


id,category_name
1,Beading & Jewelry Making
2,Fabric Decorating
3,Knitting & Crochet Supplies
4,Printmaking Supplies
5,Scrapbooking & Stamping Supplies
6,Sewing Products
7,Craft & Hobby Fabric
8,Needlework Supplies
9,"Arts, Crafts & Sewing Storage"
10,"Painting, Drawing & Art Supplies"


In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS bronze;

In [ ]:
%sql
CREATE OR REPLACE TABLE bronze.products
USING DELTA AS

SELECT *
FROM workspace.default.amazon_products;

num_affected_rows,num_inserted_rows


In [ ]:
%sql
CREATE OR REPLACE TABLE bronze.categories
USING DELTA AS

SELECT *
FROM workspace.default.amazon_categories;

num_affected_rows,num_inserted_rows


In [ ]:
%sql
SELECT COUNT(*)
FROM bronze.products;

COUNT(*)
1426337


In [ ]:
%sql
SELECT COUNT(*)
FROM bronze.categories;

COUNT(*)
248


In [ ]:
%sql
SELECT *
FROM bronze.products
LIMIT 10;

asin,title,imgUrl,productURL,stars,reviews,price,listPrice,category_id,isBestSeller,boughtInLastMonth
B00U36Z40M,Good Breath 60 Softgels,https://m.media-amazon.com/images/I/71PEEvhbE1L._AC_UL320_.jpg,https://www.amazon.com/dp/B00U36Z40M,3.9,0,8.39,0.0,45,False,0
B0BG1M2Y45,"SITHON Hard Shell Travel Case for Oral-B Pro 1000, 2000, 3000, 3500, 1500 Electric Toothbrush, Protective Shockproof Sto",https://m.media-amazon.com/images/I/71CK+asK+NL._AC_UL320_.jpg,https://www.amazon.com/dp/B0BG1M2Y45,4.3,0,7.99,0.0,45,False,0
B0B7XD6H1L,"4Pcs Replacement Switch Buttons Compatible with Dyson Hair Dryer HD01 HD02, HD03, HD04, HD08, Control Buttons Repair Acc",https://m.media-amazon.com/images/I/51Tsp7LiPbL._AC_UL320_.jpg,https://www.amazon.com/dp/B0B7XD6H1L,5.0,0,7.97,0.0,45,False,0
B010S6MCYW,Teeth Whitening Accelerator Light with 5 LED Tubes with Matched Mouth Tray | Batteries Included | Blue,https://m.media-amazon.com/images/I/71yaqslrN-L._AC_UL320_.jpg,https://www.amazon.com/dp/B010S6MCYW,3.6,0,9.99,0.0,45,False,0
B09C63JK1L,"Teeth Whitening Strips with Natural Coconut Oil, 42 Non-Slip White Strips - Professional Safe Effects Tooth Whitener Sol",https://m.media-amazon.com/images/I/81PEO2nHy0L._AC_UL320_.jpg,https://www.amazon.com/dp/B09C63JK1L,4.1,0,21.99,0.0,45,False,0
B0BYY49HKD,"Slim Silicone Aligner & Retainer Case With Vent Holes, Compatible With Invisalign, Mouth Guard Case, Night guard case, M",https://m.media-amazon.com/images/I/51Q7cfkNlZL._AC_UL320_.jpg,https://www.amazon.com/dp/B0BYY49HKD,4.5,0,7.99,0.0,45,False,0
B07NVPS7B9,Zina45 Deluxe Sonic Pulse Electric Toothbrush,https://m.media-amazon.com/images/I/71WTO2BTgBL._AC_UL320_.jpg,https://www.amazon.com/dp/B07NVPS7B9,4.0,0,250.0,0.0,45,False,0
B0BRLCPN5D,sigma times (2 Pack) Steel Bite Pro Teeth Supplement for Teeth and Gum Repair Steel Bite Original Dental Tooth Repair (1,https://m.media-amazon.com/images/I/619Kj3jlxhL._AC_UL320_.jpg,https://www.amazon.com/dp/B0BRLCPN5D,3.9,0,39.95,0.0,45,False,0
B09XJJ268B,Opalescence Whitening Bundle - Opalescence Go™ 10% Hydrogen Peroxide Prefilled Whitening Trays and Whitening Toothpaste,https://m.media-amazon.com/images/I/51StaKSbscL._AC_UL320_.jpg,https://www.amazon.com/dp/B09XJJ268B,4.1,0,94.0,0.0,45,False,0
B0BB5DK2M6,"Hiceeden 16 Pieces Toothbrush Travel Case, 4 Colors Portable Dustproof Toothbrush Holder, Breathable Toothbrush Cup Stor",https://m.media-amazon.com/images/I/717TUDoTiBL._AC_UL320_.jpg,https://www.amazon.com/dp/B0BB5DK2M6,4.8,0,14.99,0.0,45,False,0
